In [4]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================================================================
# 0. DIRECTORY PATH FIX 
# =========================================================================
# This appends the parent folder (the root of the repo) to Python's path
# so it can successfully find and import the 'models' directory from inside 
# your 'research' folder.
sys.path.append(os.path.abspath('..')) 

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp
from models.frameworks import IsoAlign

In [5]:
import os
import random
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class SleepEDF_Supervised_Dataset(Dataset):
    def __init__(self, pt_file_path, split="train"):
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict) and split in data_obj:
            data_obj = data_obj[split]
            
        if "samples" in data_obj:
            self.data = data_obj["samples"]
        else:
            self.data = data_obj.get("data", data_obj.get("X_train", []))
            
        self.data = torch.FloatTensor(self.data)
        if self.data.dim() == 2:
            self.data = self.data.unsqueeze(1)
            
        if "labels" in data_obj:
            self.labels = data_obj["labels"]
        else:
            self.labels = data_obj.get("y_data", [])
        self.labels = torch.tensor(self.labels, dtype=torch.long)
        
        self.window = torch.hann_window(128)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x_t = self.data[idx]
        x_time = x_t
        
        x_fft = torch.fft.rfft(x_t, dim=-1)
        x_fourier = torch.cat([torch.abs(x_fft), torch.angle(x_fft)], dim=0)
        
        x_stft = torch.stft(x_t, n_fft=128, hop_length=64, window=self.window, return_complex=True)
        x_wavelet = F.pad(torch.abs(x_stft)[:, :64, :], (0, 1))
        
        return x_time, x_fourier, x_wavelet, self.labels[idx]

class FourierWrapper(nn.Module):
    def __init__(self, in_channels, in_length, latent_dim):
        super().__init__()
        self.enc = FourierEncoder(in_channels=in_channels, in_length=in_length, out_channels=latent_dim)
        
        if in_length == 3000:
            self.enc.fc_abs = nn.Linear(376, 1)
            self.enc.fc_angle = nn.Linear(376, 1)

    def forward(self, x):
        return self.enc(x)

class SupervisedMultiViewFusion(nn.Module):
    def __init__(self, n_channels, time_steps, spect_freq, spect_time, latent_dim, num_classes):
        super().__init__()
        self.time_encoder = ResNet1D(
            in_channels=n_channels, base_filters=32, kernel_size=5, stride=1, groups=1,
            n_block=3, n_classes=latent_dim, downsample_gap=2, increasefilter_gap=4,
            use_do=True, backbone=True, output_dim=latent_dim
        )
        
        self.spect_encoder = UNET_2D_simp(
            input_channels=n_channels, output_channels=latent_dim, layer_n=32,
            spect_freq=spect_freq, spect_time=spect_time, kernel_size=3
        )
        self.spect_encoder.fc = nn.Linear(6, 1)
        self.spect_encoder.fc2 = nn.Linear(8, 1)

        self.ft_encoder = FourierWrapper(
            in_channels=n_channels * 2, in_length=time_steps, latent_dim=latent_dim
        )

        self.classifier = nn.Sequential(
            nn.Linear(3 * latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_t, x_w, x_f):
        _, r_t = self.time_encoder(x_t)
        _, r_f = self.spect_encoder(x_w)
        r_ft = self.ft_encoder(x_f)
        
        combined = torch.cat([r_t, r_f, r_ft], dim=1)
        return self.classifier(combined)

def train_supervised(args):
    set_seed(42)
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    os.makedirs(args.checkpoint_dir, exist_ok=True)

    full_train_dataset = SleepEDF_Supervised_Dataset(args.data_path, split="train")
    val_dataset = SleepEDF_Supervised_Dataset(args.data_path, split="val")
    test_dataset = SleepEDF_Supervised_Dataset(args.data_path, split="test")

    subset_size = int(0.1 * len(full_train_dataset))
    indices = np.arange(len(full_train_dataset))
    np.random.shuffle(indices)
    train_subset = Subset(full_train_dataset, indices[:subset_size])

    train_loader = DataLoader(train_subset, batch_size=args.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

    _, _, sample_w, _ = full_train_dataset[0]
    spect_freq = sample_w.shape[1]
    spect_time = sample_w.shape[2]

    model = SupervisedMultiViewFusion(
        n_channels=args.n_channels, time_steps=args.time_steps,
        spect_freq=spect_freq, spect_time=spect_time,
        latent_dim=args.latent_dim, num_classes=args.num_classes
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=15, factor=0.5, min_lr=1e-7)

    best_val_loss = float('inf')
    best_model_path = os.path.join(args.checkpoint_dir, f"best_supervised_fusion_{args.dataset_name}.pth")
    start_epoch = 0

    if args.resume:
        checkpoint_pattern = os.path.join(args.checkpoint_dir, "supervised_fusion_epoch_*.pth")
        checkpoint_files = glob.glob(checkpoint_pattern)
        
        if checkpoint_files:
            latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_epoch_')[-1].split('.pth')[0]))
            print(f"[*] Resuming from checkpoint: {latest_checkpoint}")
            
            checkpoint = torch.load(latest_checkpoint, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if 'best_val_loss' in checkpoint:
                best_val_loss = checkpoint['best_val_loss']
                
            start_epoch = checkpoint['epoch']
            print(f"[*] Resuming at epoch {start_epoch + 1} with Best Val Loss: {best_val_loss:.4f}")
        else:
            print("[*] No checkpoint found. Starting training from scratch.")

    for epoch in range(start_epoch, args.epochs):
        print(f"\n--- Epoch {epoch+1}/{args.epochs} ---")
        model.train()
        train_loss = 0
        total_batches = len(train_loader)

        for batch_idx, (batch_t, batch_f, batch_w, labels) in enumerate(train_loader):
            batch_t = batch_t.to(device).transpose(1, 2)
            batch_w = batch_w.to(device)
            batch_f = batch_f.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(batch_t, batch_w, batch_f)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == total_batches:
                print(f"  Batch [{batch_idx+1}/{total_batches}] | Current Loss: {loss.item():.4f}")

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_t, batch_f, batch_w, labels in val_loader:
                batch_t = batch_t.to(device).transpose(1, 2)
                batch_w = batch_w.to(device)
                batch_f = batch_f.to(device)
                labels = labels.to(device)
                
                outputs = model(batch_t, batch_w, batch_f)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for batch_t, batch_f, batch_w, labels in test_loader:
                batch_t = batch_t.to(device).transpose(1, 2)
                batch_w = batch_w.to(device)
                batch_f = batch_f.to(device)
                labels = labels.to(device)
                
                outputs = model(batch_t, batch_w, batch_f)
                _, predicted = torch.max(outputs, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()
        
        avg_train_loss = train_loss / total_batches
        avg_val_loss = val_loss / len(val_loader)
        test_accuracy = 100.0 * test_correct / test_total
        scheduler.step(avg_val_loss)

        print(f"Summary -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Test Accuracy: {test_accuracy:.2f}%")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"  [*] New best validation loss! Saved model weights to {best_model_path}")
            
        periodic_path = os.path.join(args.checkpoint_dir, f"supervised_fusion_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'best_val_loss': best_val_loss,
            'test_accuracy': test_accuracy
        }, periodic_path)
        print(f"  [*] Saved full checkpoint state to {periodic_path}")

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_t, batch_f, batch_w, labels in test_loader:
            batch_t = batch_t.to(device).transpose(1, 2)
            batch_w = batch_w.to(device)
            batch_f = batch_f.to(device)
            
            outputs = model(batch_t, batch_w, batch_f)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.numpy())

    target_names = ['Wake (0)', 'N1 (1)', 'N2 (2)', 'N3 (3)', 'REM (4)']
    print("\n" + "="*50)
    print("FINAL PER-CLASS METRICS (BEST MODEL ON TEST SET)")
    print("="*50)
    print(classification_report(all_targets, all_preds, target_names=target_names, digits=4))
    
    print("\n" + "="*50)
    print("FINAL CONFUSION MATRIX")
    print("="*50)
    print(confusion_matrix(all_targets, all_preds))

class Args:
    data_path = "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt"
    dataset_name = "sleep_multichannel_3c"
    checkpoint_dir = "checkpoints_supervised"
    device = "cuda:0" # <--- Changed device to cuda:0 here
    n_channels = 3
    time_steps = 3000
    latent_dim = 128
    num_classes = 5
    batch_size = 64
    epochs = 60
    lr = 1e-3
    weight_decay = 1e-4
    resume = True

if __name__ == "__main__":
    args = Args()
    train_supervised(args)

/tmp/ipykernel_1587252/719718681.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels, dtype=torch.long)


[*] Resuming from checkpoint: checkpoints_supervised/supervised_fusion_epoch_2.pth
[*] Resuming at epoch 3 with Best Val Loss: 55.5732

--- Epoch 3/60 ---
  Batch [10/414] | Current Loss: 0.2007
  Batch [20/414] | Current Loss: 0.2685
  Batch [30/414] | Current Loss: 0.3842
  Batch [40/414] | Current Loss: 0.2074
  Batch [50/414] | Current Loss: 0.1688
  Batch [60/414] | Current Loss: 0.2150
  Batch [70/414] | Current Loss: 0.2270
  Batch [80/414] | Current Loss: 0.1418
  Batch [90/414] | Current Loss: 0.1346
  Batch [100/414] | Current Loss: 0.2312
  Batch [110/414] | Current Loss: 0.1825
  Batch [120/414] | Current Loss: 0.1985
  Batch [130/414] | Current Loss: 0.2540
  Batch [140/414] | Current Loss: 0.2548
  Batch [150/414] | Current Loss: 0.2886
  Batch [160/414] | Current Loss: 0.2944
  Batch [170/414] | Current Loss: 0.3307
  Batch [180/414] | Current Loss: 0.1605
  Batch [190/414] | Current Loss: 0.2997
  Batch [200/414] | Current Loss: 0.1548
  Batch [210/414] | Current Loss: 

KeyboardInterrupt: 

In [2]:
print("Jai Shree Rama")

Jai Shree Rama


In [7]:
import os
import random
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class SleepEDF_Supervised_Dataset(Dataset):
    def __init__(self, pt_file_path, split="train"):
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict) and split in data_obj:
            data_obj = data_obj[split]
            
        if "samples" in data_obj:
            self.data = data_obj["samples"]
        else:
            self.data = data_obj.get("data", data_obj.get("X_train", []))
            
        self.data = torch.FloatTensor(self.data)
        if self.data.dim() == 2:
            self.data = self.data.unsqueeze(1)
            
        if "labels" in data_obj:
            self.labels = data_obj["labels"]
        else:
            self.labels = data_obj.get("y_data", [])
        self.labels = torch.tensor(self.labels, dtype=torch.long)
        
        self.window = torch.hann_window(128)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x_t = self.data[idx]
        x_time = x_t
        
        x_fft = torch.fft.rfft(x_t, dim=-1)
        x_fourier = torch.cat([torch.abs(x_fft), torch.angle(x_fft)], dim=0)
        
        x_stft = torch.stft(x_t, n_fft=128, hop_length=64, window=self.window, return_complex=True)
        x_wavelet = F.pad(torch.abs(x_stft)[:, :64, :], (0, 1))
        
        return x_time, x_fourier, x_wavelet, self.labels[idx]

class FourierWrapper(nn.Module):
    def __init__(self, in_channels, in_length, latent_dim):
        super().__init__()
        self.enc = FourierEncoder(in_channels=in_channels, in_length=in_length, out_channels=latent_dim)
        
        if in_length == 3000:
            self.enc.fc_abs = nn.Linear(376, 1)
            self.enc.fc_angle = nn.Linear(376, 1)

    def forward(self, x):
        return self.enc(x)

class SupervisedMultiViewFusion(nn.Module):
    def __init__(self, n_channels, time_steps, spect_freq, spect_time, latent_dim, num_classes):
        super().__init__()
        self.time_encoder = ResNet1D(
            in_channels=n_channels, base_filters=32, kernel_size=5, stride=1, groups=1,
            n_block=3, n_classes=latent_dim, downsample_gap=2, increasefilter_gap=4,
            use_do=True, backbone=True, output_dim=latent_dim
        )
        
        self.spect_encoder = UNET_2D_simp(
            input_channels=n_channels, output_channels=latent_dim, layer_n=32,
            spect_freq=spect_freq, spect_time=spect_time, kernel_size=3
        )
        self.spect_encoder.fc = nn.Linear(6, 1)
        self.spect_encoder.fc2 = nn.Linear(8, 1)

        self.ft_encoder = FourierWrapper(
            in_channels=n_channels * 2, in_length=time_steps, latent_dim=latent_dim
        )

        self.classifier = nn.Sequential(
            nn.Linear(3 * latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_t, x_w, x_f):
        _, r_t = self.time_encoder(x_t)
        _, r_f = self.spect_encoder(x_w)
        r_ft = self.ft_encoder(x_f)
        
        combined = torch.cat([r_t, r_f, r_ft], dim=1)
        return self.classifier(combined)

def train_supervised(args):
    set_seed(42)
    cudnn.benchmark = True
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    os.makedirs(args.checkpoint_dir, exist_ok=True)

    full_train_dataset = SleepEDF_Supervised_Dataset(args.data_path, split="train")
    val_dataset = SleepEDF_Supervised_Dataset(args.data_path, split="val")
    test_dataset = SleepEDF_Supervised_Dataset(args.data_path, split="test")

    subset_size = int(0.1 * len(full_train_dataset))
    indices = np.arange(len(full_train_dataset))
    np.random.shuffle(indices)
    train_subset = Subset(full_train_dataset, indices[:subset_size])

    train_loader = DataLoader(
        train_subset, batch_size=args.batch_size, shuffle=True, 
        drop_last=True, num_workers=4, pin_memory=True, prefetch_factor=2
    )
    val_loader = DataLoader(
        val_dataset, batch_size=args.batch_size, shuffle=False, 
        num_workers=4, pin_memory=True, prefetch_factor=2
    )
    test_loader = DataLoader(
        test_dataset, batch_size=args.batch_size, shuffle=False, 
        num_workers=4, pin_memory=True, prefetch_factor=2
    )

    _, _, sample_w, _ = full_train_dataset[0]
    spect_freq = sample_w.shape[1]
    spect_time = sample_w.shape[2]

    model = SupervisedMultiViewFusion(
        n_channels=args.n_channels, time_steps=args.time_steps,
        spect_freq=spect_freq, spect_time=spect_time,
        latent_dim=args.latent_dim, num_classes=args.num_classes
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=15, factor=0.5, min_lr=1e-7)
    scaler = torch.amp.GradScaler("cuda")

    best_val_loss = float('inf')
    best_model_path = os.path.join(args.checkpoint_dir, f"best_supervised_fusion_{args.dataset_name}.pth")
    start_epoch = 0

    if args.resume:
        checkpoint_pattern = os.path.join(args.checkpoint_dir, "supervised_fusion_epoch_*.pth")
        checkpoint_files = glob.glob(checkpoint_pattern)
        
        if checkpoint_files:
            latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_epoch_')[-1].split('.pth')[0]))
            print(f"[*] Resuming from checkpoint: {latest_checkpoint}")
            
            checkpoint = torch.load(latest_checkpoint, map_location=device)
            
            clean_state_dict = {k.replace('_orig_mod.', ''): v for k, v in checkpoint['model_state_dict'].items()}
            model.load_state_dict(clean_state_dict)
            
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if 'best_val_loss' in checkpoint:
                best_val_loss = checkpoint['best_val_loss']
                
            start_epoch = checkpoint['epoch']
            print(f"[*] Resuming at epoch {start_epoch + 1} with Best Val Loss: {best_val_loss:.4f}")
        else:
            print("[*] No checkpoint found. Starting training from scratch.")

    model = torch.compile(model)

    for epoch in range(start_epoch, args.epochs):
        print(f"\n--- Epoch {epoch+1}/{args.epochs} ---")
        model.train()
        train_loss = 0
        total_batches = len(train_loader)

        for batch_idx, (batch_t, batch_f, batch_w, labels) in enumerate(train_loader):
            batch_t = batch_t.to(device, non_blocking=True).transpose(1, 2)
            batch_w = batch_w.to(device, non_blocking=True)
            batch_f = batch_f.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast("cuda"):
                outputs = model(batch_t, batch_w, batch_f)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()

            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == total_batches:
                print(f"  Batch [{batch_idx+1}/{total_batches}] | Current Loss: {loss.item():.4f}")

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_t, batch_f, batch_w, labels in val_loader:
                batch_t = batch_t.to(device, non_blocking=True).transpose(1, 2)
                batch_w = batch_w.to(device, non_blocking=True)
                batch_f = batch_f.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                with torch.amp.autocast("cuda"):
                    outputs = model(batch_t, batch_w, batch_f)
                    loss = criterion(outputs, labels)
                val_loss += loss.item()

        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for batch_t, batch_f, batch_w, labels in test_loader:
                batch_t = batch_t.to(device, non_blocking=True).transpose(1, 2)
                batch_w = batch_w.to(device, non_blocking=True)
                batch_f = batch_f.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                with torch.amp.autocast("cuda"):
                    outputs = model(batch_t, batch_w, batch_f)
                _, predicted = torch.max(outputs, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()
        
        avg_train_loss = train_loss / total_batches
        avg_val_loss = val_loss / len(val_loader)
        test_accuracy = 100.0 * test_correct / test_total
        scheduler.step(avg_val_loss)

        print(f"Summary -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Test Accuracy: {test_accuracy:.2f}%")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"  [*] New best validation loss! Saved model weights to {best_model_path}")
            
        periodic_path = os.path.join(args.checkpoint_dir, f"supervised_fusion_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'best_val_loss': best_val_loss,
            'test_accuracy': test_accuracy
        }, periodic_path)
        print(f"  [*] Saved full checkpoint state to {periodic_path}")

    best_weights = torch.load(best_model_path, map_location=device)
    clean_best_weights = {k.replace('_orig_mod.', ''): v for k, v in best_weights.items()}
    
    if hasattr(model, "_orig_mod"):
        model._orig_mod.load_state_dict(clean_best_weights)
    else:
        model.load_state_dict(clean_best_weights)
        
    model.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_t, batch_f, batch_w, labels in test_loader:
            batch_t = batch_t.to(device, non_blocking=True).transpose(1, 2)
            batch_w = batch_w.to(device, non_blocking=True)
            batch_f = batch_f.to(device, non_blocking=True)
            
            with torch.amp.autocast("cuda"):
                outputs = model(batch_t, batch_w, batch_f)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.numpy())

    target_names = ['Wake (0)', 'N1 (1)', 'N2 (2)', 'N3 (3)', 'REM (4)']
    print("\n" + "="*50)
    print("FINAL PER-CLASS METRICS (BEST MODEL ON TEST SET)")
    print("="*50)
    print(classification_report(all_targets, all_preds, target_names=target_names, digits=4))
    
    print("\n" + "="*50)
    print("FINAL CONFUSION MATRIX")
    print("="*50)
    print(confusion_matrix(all_targets, all_preds))

class Args:
    data_path = "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt"
    dataset_name = "sleep_multichannel_3c"
    checkpoint_dir = "checkpoints_supervised"
    device = "cuda:0"
    n_channels = 3
    time_steps = 3000
    latent_dim = 128
    num_classes = 5
    batch_size = 64
    epochs = 60
    lr = 1e-3
    weight_decay = 1e-4
    resume = True

if __name__ == "__main__":
    args = Args()
    train_supervised(args)

/tmp/ipykernel_1587252/2798387613.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels, dtype=torch.long)


[*] Resuming from checkpoint: checkpoints_supervised/supervised_fusion_epoch_2.pth
[*] Resuming at epoch 3 with Best Val Loss: 55.5732

--- Epoch 3/60 ---


W0706 23:23:12.827000 1587252 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


  Batch [10/414] | Current Loss: 0.2056
  Batch [20/414] | Current Loss: 0.2403
  Batch [30/414] | Current Loss: 0.3855
  Batch [40/414] | Current Loss: 0.2062
  Batch [50/414] | Current Loss: 0.1648
  Batch [60/414] | Current Loss: 0.1924
  Batch [70/414] | Current Loss: 0.2416
  Batch [80/414] | Current Loss: 0.1609
  Batch [90/414] | Current Loss: 0.1705
  Batch [100/414] | Current Loss: 0.2280
  Batch [110/414] | Current Loss: 0.1554
  Batch [120/414] | Current Loss: 0.2413
  Batch [130/414] | Current Loss: 0.2247
  Batch [140/414] | Current Loss: 0.2521
  Batch [150/414] | Current Loss: 0.3156
  Batch [160/414] | Current Loss: 0.3013
  Batch [170/414] | Current Loss: 0.2572
  Batch [180/414] | Current Loss: 0.1390
  Batch [190/414] | Current Loss: 0.3470
  Batch [200/414] | Current Loss: 0.1818
  Batch [210/414] | Current Loss: 0.2777
  Batch [220/414] | Current Loss: 0.2660
  Batch [230/414] | Current Loss: 0.3033
  Batch [240/414] | Current Loss: 0.2552
  Batch [250/414] | Curre